# 08 Forecast Plotly Diagnostics

This notebook builds interactive Plotly HTML diagnostics from existing forecast CSV files. It does not train, refit, or re-estimate any model.

Each figure shows the full actual series and overlays model forecasts only for the test period. The HTML files can be zoomed and panned during seminar discussion.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "Transport_amount_project" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "parcel_volume_connected.csv"
PREDICTIONS_DIR = PROJECT_ROOT / "output" / "forecasts" / "predictions"
HTML_DIR = PROJECT_ROOT / "output" / "forecasts" / "html"
HTML_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Predictions dir:", PREDICTIONS_DIR)
print("HTML dir:", HTML_DIR)

Project root: C:\Users\fugat\Desktop\python_project\Transport_amount_project
Predictions dir: C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\predictions
HTML dir: C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\html


## 1. Load Actuals And Forecast CSVs

The actual series is loaded from `data/processed/parcel_volume_connected.csv`. Forecast files are loaded only when they exist; missing files are skipped instead of stopping the notebook.

In [2]:
actual = pd.read_csv(DATA_PATH)
if "date" in actual.columns:
    actual["date"] = pd.to_datetime(actual["date"])
elif "label" in actual.columns:
    actual["date"] = pd.to_datetime(actual["label"])
else:
    actual["date"] = pd.to_datetime(dict(year=actual["year"], month=actual["month"], day=1))

actual = actual.sort_values("date")[["date", "number_parcels"]].copy()
actual["number_parcels"] = pd.to_numeric(actual["number_parcels"], errors="coerce")

def read_prediction(filename: str) -> pd.DataFrame | None:
    path = PREDICTIONS_DIR / filename
    if not path.exists():
        print(f"skip: {filename} not found")
        return None
    frame = pd.read_csv(path)
    frame["date"] = pd.to_datetime(frame["date"])
    frame["cutoff"] = pd.to_datetime(frame["cutoff"])
    frame["y_true"] = pd.to_numeric(frame["y_true"], errors="coerce")
    frame["y_pred"] = pd.to_numeric(frame["y_pred"], errors="coerce")
    return frame

prediction_files = {
    "fixed_a_naive": "fixed_a_naive.csv",
    "fixed_b_naive": "fixed_b_naive.csv",
    "fixed_a_sarimax": "fixed_a_sarimax.csv",
    "fixed_b_sarimax": "fixed_b_sarimax.csv",
    "fixed_a_ssm": "fixed_a_ssm.csv",
    "fixed_b_ssm": "fixed_b_ssm.csv",
    "fixed_a_prophet": "fixed_a_prophet.csv",
    "fixed_b_prophet": "fixed_b_prophet.csv",
    "fixed_b_autoformer_lite": "fixed_b_autoformer_lite.csv",
    "fixed_b_prophet_regressors": "fixed_b_prophet_regressors.csv",
}

predictions = {key: read_prediction(filename) for key, filename in prediction_files.items()}
predictions = {key: value for key, value in predictions.items() if value is not None}

summary = []
for key, frame in predictions.items():
    summary.append(
        {
            "key": key,
            "models": ", ".join(sorted(frame["model"].unique())),
            "split": ", ".join(sorted(frame["split"].unique())),
            "forecast_type": ", ".join(sorted(frame["forecast_type"].unique())),
            "start": frame["date"].min().date(),
            "end": frame["date"].max().date(),
            "rows": len(frame),
        }
    )

display(pd.DataFrame(summary))

,key,models,split,forecast_type,start,end,rows
0,fixed_a_naive,"naive, seasonal_naive",fixed_a,unconditional,2020-01-01,2021-12-01,48
1,fixed_b_naive,"naive, seasonal_naive",fixed_b,unconditional,2024-01-01,2026-02-01,52
2,fixed_a_sarimax,"sarima, sarimax",fixed_a,"conditional, unconditional",2020-01-01,2021-12-01,48
3,fixed_b_sarimax,"sarima, sarimax",fixed_b,"conditional, unconditional",2024-01-01,2026-02-01,52
4,fixed_a_ssm,ssm,fixed_a,conditional,2020-01-01,2021-12-01,24
5,fixed_b_ssm,ssm,fixed_b,conditional,2024-01-01,2026-02-01,26
6,fixed_a_prophet,prophet,fixed_a,unconditional,2020-01-01,2021-12-01,24
7,fixed_b_prophet,prophet,fixed_b,unconditional,2024-01-01,2026-02-01,26
8,fixed_b_autoformer_lite,autoformer_lite,fixed_b,unconditional,2024-01-01,2026-02-01,26
9,fixed_b_prophet_regressors,prophet_regressors,fixed_b,conditional,2024-01-01,2026-02-01,26


## 2. Plot Helper

The black line is the full actual series. Forecast traces are drawn only over the test period. Cutoff dates are shown as vertical dashed lines. The legend can be used to show or hide individual models.

In [3]:
def model_frame(source_key: str, model: str, display_name: str) -> pd.DataFrame:
    frame = predictions[source_key]
    selected = frame[frame["model"] == model].copy()
    selected["display_model"] = display_name
    return selected


def make_plot(title: str, forecast_frames: list[pd.DataFrame], output_path: Path) -> go.Figure:
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=actual["date"],
            y=actual["number_parcels"],
            mode="lines",
            name="actual",
            line=dict(color="black", width=2),
            hovertemplate="date=%{x|%Y-%m}<br>actual=%{y:,.0f}<br>model=actual<extra></extra>",
        )
    )

    palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]
    cutoff_dates = []
    for idx, frame in enumerate(forecast_frames):
        if frame.empty:
            continue
        label = frame["display_model"].iloc[0]
        cutoff_dates.extend(frame["cutoff"].dropna().unique().tolist())
        fig.add_trace(
            go.Scatter(
                x=frame["date"],
                y=frame["y_pred"],
                mode="lines+markers",
                name=label,
                line=dict(color=palette[idx % len(palette)], width=2),
                marker=dict(size=5),
                customdata=frame[["display_model", "y_true", "horizon"]].to_numpy(),
                hovertemplate=(
                    "date=%{x|%Y-%m}<br>"
                    "predicted=%{y:,.0f}<br>"
                    "actual=%{customdata[1]:,.0f}<br>"
                    "horizon=%{customdata[2]}<br>"
                    "model=%{customdata[0]}<extra></extra>"
                ),
            )
        )

    for cutoff in sorted(pd.to_datetime(pd.Series(cutoff_dates)).dropna().unique()):
        cutoff_ts = pd.Timestamp(cutoff)
        cutoff_x = cutoff_ts.strftime("%Y-%m-%d")
        fig.add_shape(
            type="line",
            x0=cutoff_x,
            x1=cutoff_x,
            y0=0,
            y1=1,
            xref="x",
            yref="paper",
            line=dict(color="gray", width=1, dash="dash"),
        )
        fig.add_annotation(
            x=cutoff_x,
            y=1,
            xref="x",
            yref="paper",
            text=f"cutoff {cutoff_ts.strftime('%Y-%m')}",
            showarrow=False,
            yanchor="bottom",
            font=dict(color="gray", size=11),
        )

    fig.update_layout(
        title=title,
        xaxis_title="Date",
        yaxis_title="number_parcels",
        hovermode="x unified",
        template="plotly_white",
        legend_title="Series",
        width=1100,
        height=620,
    )
    fig.update_xaxes(rangeslider_visible=True)
    fig.write_html(output_path, include_plotlyjs="cdn", full_html=True)
    print(f"saved: {output_path}")
    return fig

## 3. fixed A Unconditional

Included series: actual, seasonal naive, SARIMA, and Prophet.

No prediction CSV exists for `sarima_grid_best`, so this figure uses fixed-order `sarima_fixed` from `fixed_a_sarimax.csv` as the available substitute.

In [4]:
fixed_a_unconditional_frames = [
    model_frame("fixed_a_naive", "seasonal_naive", "seasonal_naive"),
    model_frame("fixed_a_sarimax", "sarima", "sarima_fixed"),
    model_frame("fixed_a_prophet", "prophet", "prophet"),
]

fig_fixed_a_unconditional = make_plot(
    "fixed A unconditional forecasts: actual vs predicted",
    fixed_a_unconditional_frames,
    HTML_DIR / "fixed_a_unconditional_forecasts.html",
)
fig_fixed_a_unconditional.show()

saved: C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\html\fixed_a_unconditional_forecasts.html


## 4. fixed B Unconditional

Included series: actual, seasonal naive, SARIMA, Prophet, and autoformer_lite.

No prediction CSV exists for `sarima_grid_best`, so this figure uses fixed-order `sarima_fixed` from `fixed_b_sarimax.csv` as the available substitute. `autoformer_lite` is an Autoformer-inspired / decomposition Transformer baseline, not a strict Autoformer implementation.

In [5]:
fixed_b_unconditional_frames = [
    model_frame("fixed_b_naive", "seasonal_naive", "seasonal_naive"),
    model_frame("fixed_b_sarimax", "sarima", "sarima_fixed"),
    model_frame("fixed_b_prophet", "prophet", "prophet"),
    model_frame("fixed_b_autoformer_lite", "autoformer_lite", "autoformer_lite"),
]

fig_fixed_b_unconditional = make_plot(
    "fixed B unconditional forecasts: actual vs predicted",
    fixed_b_unconditional_frames,
    HTML_DIR / "fixed_b_unconditional_forecasts.html",
)
fig_fixed_b_unconditional.show()

saved: C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\html\fixed_b_unconditional_forecasts.html


## 5. fixed B Conditional

Included series: actual, SARIMAX, SSM, and Prophet regressors.

No prediction CSV exists for `sarimax_grid_best`, so this figure uses fixed-order `sarimax_fixed` from `fixed_b_sarimax.csv` as the available substitute.

These are conditional forecasts because test-period exogenous dummies are treated as known. They should be interpreted separately from the unconditional forecasts.

In [6]:
fixed_b_conditional_frames = [
    model_frame("fixed_b_sarimax", "sarimax", "sarimax_fixed"),
    model_frame("fixed_b_ssm", "ssm", "ssm_conditional"),
    model_frame("fixed_b_prophet_regressors", "prophet_regressors", "prophet_regressors"),
]

fig_fixed_b_conditional = make_plot(
    "fixed B conditional forecasts: actual vs predicted",
    fixed_b_conditional_frames,
    HTML_DIR / "fixed_b_conditional_forecasts.html",
)
fig_fixed_b_conditional.show()

saved: C:\Users\fugat\Desktop\python_project\Transport_amount_project\output\forecasts\html\fixed_b_conditional_forecasts.html


## 6. Saved HTML Files And Substitutions

Saved files:

- `output/forecasts/html/fixed_a_unconditional_forecasts.html`
- `output/forecasts/html/fixed_b_unconditional_forecasts.html`
- `output/forecasts/html/fixed_b_conditional_forecasts.html`

Substitutions:

- `sarima_grid_best` prediction CSV was not available, so `sarima_fixed` was used.
- `sarimax_grid_best` prediction CSV was not available, so `sarimax_fixed` was used.

This notebook only visualizes existing forecast CSV files. It does not refit or retrain any model.